In [1]:
import pandas as pd
from sqlalchemy import create_engine,URL,text

In [2]:
def data_extraction():
    try:
        connection_url = URL.create(
            drivername="mysql+pymysql",
            username="root",
            password="Shantanu@2026",
            host="localhost",
            port=3306,
            database="data_cleansing"
        )

        engine = create_engine(connection_url)

        sales = pd.read_sql("SELECT * FROM sales_source", engine)

        return sales

    except Exception as e:
        print(f"Error: {e}")

In [3]:
df=data_extraction()

In [4]:
df

,sale_id,customer_name,product_name,quantity,unit_price,total_amount,sale_date,city
0,1,Rahul Sharma,Laptop,2.0,55000.0,110000.0,2026-01-10,Pune
1,2,Priya Patel,Mouse,3.0,500.0,1500.0,2026-01-12,Mumbai
2,3,Amit Verma,Keyboard,-2.0,1200.0,-2400.0,2026-01-15,Delhi
3,4,NaN,Monitor,1.0,12000.0,12000.0,2026-01-18,Pune
4,5,Sneha Joshi,Headphones,2.0,-1500.0,-3000.0,2026-01-20,Mumbai
5,6,Karan Mehta,NaN,1.0,25000.0,25000.0,2026-01-22,Delhi
6,7,Neha Singh,Printer,NaN,8000.0,8000.0,2026-01-25,Pune
7,8,Rohit Kumar,Tablet,2.0,15000.0,25000.0,2026-01-27,Mumbai
8,9,Anjali Desai,Smartphone,1.0,30000.0,30000.0,None,Pune
9,10,Vikas Rao,Laptop,1.0,55000.0,55000.0,2026-01-30,NaN


In [24]:

def data_transformation(df):

    # Standardize column names
    df.columns = df.columns.str.lower()

    # String cleansing
    for i in df.columns:
        if df[i].dtype == 'str':
            df[i] = df[i].fillna('Unknown')
            df[i] = df[i].str.strip()
            df.loc[df[i] == "", i] = "Unknown"

    df["quantity"] = abs(df["quantity"])
    df["unit_price"] = abs(df["unit_price"])
    df["total_amount"] = abs(df["total_amount"])

    # Quantity: handle invalid values
    df.loc[df["quantity"] <= 0, "quantity"] = None

    # Unit price: recover missing values
    df.loc[df["unit_price"] <= 0, "unit_price"] = None

    df["unit_price"] = df["unit_price"].fillna(
        df["total_amount"] / df["quantity"]
    )

    df["unit_price"] = df["unit_price"].fillna(
        df["unit_price"].mean()
    )

    # Quantity: recover missing values
    df["quantity"] = df["quantity"].fillna(
        df["total_amount"] / df["unit_price"]
    )

    df["quantity"] = df["quantity"].fillna(1)

    # Total amount: recalculate for missing or invalid values
    df["total_amount"] = df["quantity"] * df["unit_price"]

    return df


In [25]:
new_df = data_transformation(df)

In [26]:
new_df.isna().sum()

sale_id          0
customer_name    0
product_name     0
quantity         0
unit_price       0
total_amount     0
sale_date        0
city             0
dtype: int64

In [27]:
new_df

,sale_id,customer_name,product_name,quantity,unit_price,total_amount,sale_date,city
0,1,Rahul Sharma,Laptop,2.0,55000.0,110000.0,Unknown,Pune
1,2,Priya Patel,Mouse,3.0,500.0,1500.0,Unknown,Mumbai
2,3,Amit Verma,Keyboard,2.0,1200.0,2400.0,Unknown,Delhi
3,4,Unknown,Monitor,1.0,12000.0,12000.0,Unknown,Pune
4,5,Sneha Joshi,Headphones,2.0,1500.0,3000.0,Unknown,Mumbai
5,6,Karan Mehta,Unknown,1.0,25000.0,25000.0,Unknown,Delhi
6,7,Neha Singh,Printer,1.0,8000.0,8000.0,Unknown,Pune
7,8,Rohit Kumar,Tablet,2.0,15000.0,30000.0,Unknown,Mumbai
8,9,Anjali Desai,Smartphone,1.0,30000.0,30000.0,Unknown,Pune
9,10,Vikas Rao,Laptop,1.0,55000.0,55000.0,Unknown,Unknown
